# Démonstration de l'outil de recommandation

Ce notebook exécute `recommander` (moteur basé sur la grille de codage) sur trois scénarios : les deux cas d'usage du mémoire et une page de synthèse centrée sur les KPI.

Les règles (Apriori, fréquences conditionnelles, fallback) sont calculées au chargement à partir de `data/processed/grille_analyse_anonymisee.csv`. Pour l'interface interactive, voir `02_interactive_tool.ipynb`.

In [1]:
import sys
from pathlib import Path

import pandas as pd

src_path = Path.cwd().resolve().parent / 'src'
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from reco_tool import recommander
from reco_tool.recommender import get_rules

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 140)

stats = get_rules()['stats']
print(f"{stats['pages']} pages · {stats['itemsets_frequents']} itemsets fréquents · {stats['regles_apriori']} règles Apriori")

34 pages · 119 itemsets fréquents · 27 règles Apriori


## Scénarios

Les valeurs de niveau s'écrivent `Faible`, `Moderee` ou `Forte`, comme dans la grille (les accents et la casse sont acceptés).

In [2]:
scenarios = [
    {
        'scenario': 'Cas 1 - page dense avec table de détail',
        'kpi_present': 'Oui', 'nb_kpi': 4, 'serie_present': 'Non', 'table_present': 'Oui',
        'alerte_present': 'Non', 'nb_filtres': 4,
        'densite': 'Forte', 'hierarchie_couleur': 'Faible', 'hierarchie_taille': 'Faible',
    },
    {
        'scenario': 'Cas 2 - dashboard de suivi temporel',
        'kpi_present': 'Oui', 'nb_kpi': 4, 'serie_present': 'Oui', 'table_present': 'Oui',
        'alerte_present': 'Non', 'nb_filtres': 3,
        'densite': 'Moderee', 'hierarchie_couleur': 'Forte', 'hierarchie_taille': 'Forte',
    },
    {
        'scenario': 'Cas 3 - page de synthèse KPI',
        'kpi_present': 'Oui', 'nb_kpi': 7, 'serie_present': 'Non', 'table_present': 'Non',
        'alerte_present': 'Non', 'nb_filtres': 2,
        'densite': 'Moderee', 'hierarchie_couleur': 'Moderee', 'hierarchie_taille': 'Faible',
    },
]

INPUTS = [
    'kpi_present', 'nb_kpi', 'serie_present', 'table_present', 'alerte_present',
    'nb_filtres', 'densite', 'hierarchie_couleur', 'hierarchie_taille',
]

## Résultats détaillés

Pour chaque recommandation : la valeur, la confiance issue de la grille (en %) et la source (`frequentiel`, `apriori`, `grille`, `elements` ou `fallback`).

In [3]:
def formater(info):
    texte = str(info['valeur'])
    if info['confiance'] is not None:
        return f"{texte} ({info['confiance']:.1f} %, {info['source']})"
    return f"{texte} ({info.get('note', info['source'])})"


for scenario in scenarios:
    entrees = {cle: scenario[cle] for cle in INPUTS}
    reco = recommander(**entrees)
    print('=' * 78)
    print(scenario['scenario'])
    print('-' * 78)
    print('Entrées :', entrees)
    for cible, info in reco.items():
        print(f'  {cible:34s} -> {formater(info)}')
    print()

Cas 1 - page dense avec table de détail
------------------------------------------------------------------------------
Entrées : {'kpi_present': 'Oui', 'nb_kpi': 4, 'serie_present': 'Non', 'table_present': 'Oui', 'alerte_present': 'Non', 'nb_filtres': 4, 'densite': 'Forte', 'hierarchie_couleur': 'Faible', 'hierarchie_taille': 'Faible'}
  Tables_detail_emplacement          -> centre (60.0 %, frequentiel)
  Affordances_actions                -> Forte (75.0 %, frequentiel)
  Interactions_types                 -> slicers (70.0 %, frequentiel)
  Risques_surcharge                  -> Oui (100.0 %, apriori)
  KPI_emplacement                    -> ligne_supérieure (Optimal)
  Visuels_recommandes                -> KPI_card, table / matrix (elements)
  Layout_dominant                    -> Z_pattern (fallback)

Cas 2 - dashboard de suivi temporel
------------------------------------------------------------------------------
Entrées : {'kpi_present': 'Oui', 'nb_kpi': 4, 'serie_present': 'Oui', 't

## Tableau comparatif

Une cible absente d'un scénario signifie qu'aucune règle de la grille ne couvre cette configuration.

In [4]:
lignes = []
for scenario in scenarios:
    reco = recommander(**{cle: scenario[cle] for cle in INPUTS})
    ligne = {'scenario': scenario['scenario']}
    ligne.update({cible: info['valeur'] for cible, info in reco.items()})
    lignes.append(ligne)

pd.DataFrame(lignes).set_index('scenario').T

scenario,Cas 1 - page dense avec table de détail,Cas 2 - dashboard de suivi temporel,Cas 3 - page de synthèse KPI
Tables_detail_emplacement,centre,bas,centre
Affordances_actions,Forte,Forte,Moderee
Interactions_types,slicers,slicers,slicers
Risques_surcharge,Oui,NaN,NaN
KPI_emplacement,ligne_supérieure,ligne_supérieure,ligne_supérieure
Visuels_recommandes,"KPI_card, table / matrix","KPI_card, line_chart / area_chart, table / matrix",KPI_card
Layout_dominant,Z_pattern,Hybride,Z_pattern
Filtres_emplacement,NaN,integres_visuels,ligne_supérieur
Types_visuels,NaN,table,table


## Lecture des résultats

- Le risque de surcharge vient des règles Apriori (confiance minimale de 90 %). Les emplacements viennent des fréquences conditionnelles (au moins 5 pages et 60 %).
- Quand aucune règle ne couvre le cas, le layout est proposé par une logique de fallback, sans pourcentage de confiance.
- Ces règles décrivent des associations observées sur 34 pages d'un même contexte, pas des relations de cause à effet.